# 第 19 章：状态空间与 Kalman Filter

本 Notebook 从 code/ch19_state_space.py 复用工作流与 src/quant_course/state_space.py 的核心滤波器。它在内存中复算手算例、固定种子模拟和冻结的 French 历史快照；不估计初值、Q 或 R，不实现 smoother，也不写回任何 CSV、JSON、图或本 Notebook。

真实资料采用月度 NoDur 简单收益减 RF，市场输入为 Mkt-RF。当前修订快照只支持描述性重算，不构成 PIT 回测或可交易收益证据。

**离线构造数据教学示例。** French 数据未随包提供，相关使用权与实证执行授权尚未确立，因此本 Notebook 不运行真实数据分支。构造示例与静态教学图均不代表完整实证复现。


In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch19_state_space_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

from pathlib import Path
import sys

sys.path.insert(0, str(ROOT / 'code'))
import ch19_state_space as workflow
from quant_course.state_space import kalman_filter
import numpy as np
from IPython.display import Image, display
print('Project root:', ROOT)

## 标量更新：先验 → 创新 → 增益 → 后验

beta 先验均值为 1、方差为 .05；当月市场超额收益 .02、资产超额收益 .03、观测方差 .0001。观测均值 .02，创新 .01，创新方差 .00012。增益 8.3333 的单位是 beta / 收益，不是概率。收益输入采用月度小数。

In [ ]:
scalar = kalman_filter(
    [.03], [[.02]], initial_state=[1.0], initial_covariance=[[.04]],
    transition=[[1.0]], process_covariance=[[.01]], observation_variance=.0001,
)
expected = [1.0, .05, .01, .00012, 8.333333333333333, 1.0833333333333333, .04166666666666667]
actual = [scalar.predicted_states[0, 0], scalar.predicted_covariances[0, 0, 0],
          scalar.innovations[0], scalar.innovation_variances[0], scalar.gains[0, 0],
          scalar.filtered_states[0, 0], scalar.filtered_covariances[0, 0, 0]]
np.testing.assert_allclose(actual, expected)
print('prior beta, prior variance, innovation, S, K, posterior beta, posterior variance')
print(actual)

## 二维 alpha/beta 条件正态核对

以 alpha 均值 0、beta 均值 1 为先验，协方差为 diag(0.0004, 0.04)。设计为 [1, 0.02]，月度小数资产超额收益为 0.03，观测噪声方差为 0.0001。手算创新为 0.01、创新方差为 0.000516，后验均值约为 [0.007752, 1.015504]。下面把条件 Gaussian 公式与核心过滤器比较。

In [ ]:
P = np.diag([.0004, .04])
h = np.array([1.0, .02])
prior_mean = np.array([0.0, 1.0])
y, R = .03, .0001
S = float(h @ P @ h + R)
cross = P @ h
innovation = y - h @ prior_mean
gain = cross / S
conditional_mean = prior_mean + gain * innovation
conditional_covariance = P - np.outer(cross, cross) / S
two_state = kalman_filter([y], [h], initial_state=prior_mean,
                          initial_covariance=P, transition=np.eye(2),
                          process_covariance=np.zeros((2, 2)), observation_variance=R)
np.testing.assert_allclose(S, .000516)
np.testing.assert_allclose(innovation, .01)
np.testing.assert_allclose(two_state.innovation_variances[0], S)
np.testing.assert_allclose(two_state.gains[0], gain)
np.testing.assert_allclose(two_state.filtered_states[0], conditional_mean)
np.testing.assert_allclose(two_state.filtered_covariances[0], conditional_covariance)
np.testing.assert_allclose(conditional_mean, [.0077519379845, 1.015503875969])
assert np.linalg.eigvalsh(conditional_covariance).min() >= 0
print('innovation =', innovation, '\nS =', S, '\nK =', gain,
      '\nposterior mean =', conditional_mean,
      '\nposterior covariance =\n', conditional_covariance)

## 固定种子机制与 French 描述性案例

模拟包含已知 alpha 和 beta 潜状态，并对比真实 Q 与 beta 过程方差误设为零的过滤器。单条路径只帮助看算法与误设的作用，不是未来预测优劣证据。

French 案例使用固定教学初值、Q 和 R，递归过滤 1926-07 至 2026-08 的共同月度历史。参数不由全样本估计；当前修订数据也不是 PIT 历史。

In [ ]:
simulated, simulation_filter, simulation_metrics = workflow.simulate_known_latent()
print("CONSTRUCTED", simulation_metrics)
simulated.to_csv(RUN_DIR / "constructed_latent.csv")
simulation_filter.to_csv(RUN_DIR / "constructed_filter.csv")
assert len(simulated) == workflow.SIMULATION_N
print("French filter branch SKIPPED: data rights unknown")


## Filter 与 smoother 的信息差

predicted_state[t] 只用截至 t-1 的观测；filtered_state[t] 纳入 t 月观测。概念上的 smoothed_state[t] 会条件于整段样本，包含 t 以后的信息；本项目核心没有 smoother 实现。下图由 CLI 预先生成，Notebook 只读展示，不重新生成或覆盖图文件。

In [ ]:
figure_names = [
    'ch19_simulated_latent_beta.png',
    'ch19_simulated_innovation_update.png',
]
for name in figure_names:
    path = ROOT / 'fixtures/figures/a4' / name
    assert path.is_file(), f'Missing pre-generated figure: {path}'
    print(name)
    display(Image(filename=str(path)))